# Чистовик: константный прогноз

Заготовка, которая уже проходит весь путь: читает `data/`, считает CV по фолдам, пишет `submission.csv` и `cv.json` и печатает `cv.json` последней строкой. Замените прогноз и разбиение на свои.

In [46]:
import json, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

In [ ]:
sys.path.insert(0, "data")
from metric_cv import quantile_quality, cv_json

tr = pd.read_csv("data/train.csv")
sample = pd.read_csv("data/sample_submission.csv")
y = tr["daily_orders"].to_numpy(float)
QUANTILES = (0.1, 0.5, 0.9)

In [48]:



def predict(y_train, n):
    """Константный прогноз: квантили спроса по всему train. Замените на модель."""
    
    q = np.quantile(y_train, QUANTILES)
    return np.tile(q, (n, 1))
    


In [49]:
# CV: четыре случайных фолда — замените разбиением, которое похоже на тест
splits = list(KFold(4, shuffle=True, random_state=0).split(tr))
oof = np.zeros((len(tr), 3))
for a, b in splits:
    oof[b] = predict(y[a], len(b))
folds = [100 * quantile_quality(y[b], oof[b, 0], oof[b, 1], oof[b, 2], tr["city"].iloc[b]) for _, b in splits]

In [50]:
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error

tr = pd.read_csv("data/train.csv")


X = tr.drop(columns=['daily_orders','id'])
y = tr['daily_orders']

kf = KFold(n_splits=5,shuffle=False)
kf.get_n_splits()
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()








for i, (train_index, test_index) in enumerate(kf.split(X)):
    cb = CatBoostRegressor(early_stopping_rounds=50,cat_features=categorical_cols,verbose=False)
    cb.fit(X.iloc[train_index],y.iloc[train_index])
    y_pred = cb.predict(X.iloc[test_index])

    rmse = root_mean_squared_error(y.iloc[test_index],y_pred)
    print(f"Fold {i}:")
    print(f"  Test:     {rmse}")


/tmp/ipykernel_179651/3406214350.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()


Fold 0:
  Test:     33.062061487770336
Fold 1:
  Test:     32.86259998118738
Fold 2:
  Test:     29.84118344113127
Fold 3:
  Test:     29.268365466917196
Fold 4:
  Test:     28.705205761007733


In [55]:
# посылка: колонки и строки как в sample_submission.csv, 0 <= p10 <= p50 <= p90 <= 5000
te = pd.read_csv("data/test.csv")
te_without = te.drop(columns=['id'])
y_pred_to_subm = cb.predict(te_without)

test = np.round(predict(y_pred_to_subm, len(te)), 1)
sub = pd.DataFrame({"id": te["id"],
                    "predicted_daily_orders_p10": test[:, 0],
                    "predicted_daily_orders_p50": test[:, 1],
                    "predicted_daily_orders_p90": test[:, 2]})
assert list(sub.columns) == list(sample.columns) and len(sub) == len(sample)
assert set(sub["id"]) == set(sample["id"]) and (sub.iloc[:, 1:].to_numpy() >= 0).all()
sub.to_csv("submission.csv", index=False)

cv = cv_json("constant", [f / 100 for f in folds])   # пишет cv.json
print(json.dumps(cv))                                  # последняя строка — для проверки воспроизводимости


{"name": "constant", "metric": "quantile_quality_x100", "folds": [0.0, 0.0, 0.0, 0.0], "cv": 0.0, "std": 0.0}
